# Iso-error geometry explorer

Interactively compare covariance-weighted prediction ellipses with Euclidean control circles. The default uses $\Sigma=\operatorname{diag}(1,0.04)$, so each prediction contour has a $5{:}1$ axis ratio.

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
import numpy as np
from IPython.display import Image, clear_output, display

cwd = Path.cwd().resolve()
if (cwd / 'scripts' / 'plot_iso_error_geometry.py').exists():
    REPO_ROOT = cwd
elif (cwd.parent / 'scripts' / 'plot_iso_error_geometry.py').exists():
    REPO_ROOT = cwd.parent
else:
    raise RuntimeError('Launch this notebook from the repository root or notebooks/.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.plot_iso_error_geometry import make_figure

PREVIEW_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'iso_error_geometry_explorer'
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_PATH = PREVIEW_DIR / 'preview.png'

## Interactive preview

The preview updates after releasing a slider. Prediction-error levels are sorted before plotting; $z=0$ is always included as the optimal control contour.

In [ ]:
layout = widgets.ToggleButtons(
    options=[('Overlay', 'overlay'), ('Side by side', 'side-by-side')],
    value='overlay', description='Layout')
label_mode = widgets.ToggleButtons(
    options=[('Compact + caption', 'caption'), ('Details on curves', 'inline')],
    value='caption', description='Labels')
epsilon = widgets.FloatLogSlider(
    value=0.04, base=10, min=-3, max=-0.3, step=0.05,
    description='epsilon', readout_format='.3g', continuous_update=False)
beta1 = widgets.FloatSlider(
    value=1.1, min=-2.0, max=2.0, step=0.05,
    description='beta* 1', continuous_update=False)
beta2 = widgets.FloatSlider(
    value=0.65, min=-2.0, max=2.0, step=0.05,
    description='beta* 2', continuous_update=False)
error1 = widgets.FloatLogSlider(
    value=0.01, base=10, min=-3, max=0, step=0.05,
    description='E gen 1', readout_format='.3g', continuous_update=False)
error2 = widgets.FloatLogSlider(
    value=0.04, base=10, min=-3, max=0, step=0.05,
    description='E gen 2', readout_format='.3g', continuous_update=False)
error3 = widgets.FloatLogSlider(
    value=0.09, base=10, min=-3, max=0, step=0.05,
    description='E gen 3', readout_format='.3g', continuous_update=False)
z_low = widgets.FloatSlider(
    value=-1/3, min=-0.9, max=-0.02, step=0.01,
    description='z low', readout_format='.2f', continuous_update=False)
z_high = widgets.FloatSlider(
    value=1.0, min=0.02, max=2.0, step=0.02,
    description='z high', readout_format='.2f', continuous_update=False)

controls = [layout, label_mode, epsilon, beta1, beta2, error1, error2, error3, z_low, z_high]
preview_output = widgets.Output()

def current_parameters():
    return dict(
        epsilon=float(epsilon.value),
        beta_star=np.array([beta1.value, beta2.value], dtype=float),
        prediction_errors=tuple(sorted([error1.value, error2.value, error3.value])),
        z_values=(float(z_low.value), 0.0, float(z_high.value)),
        layout=layout.value,
        label_mode=label_mode.value,
    )

def render_preview(change=None):
    with preview_output:
        clear_output(wait=True)
        try:
            make_figure(
                **current_parameters(),
                output=PREVIEW_PATH,
                pdf_output=None,
                table_output=None,
            )
            display(Image(filename=str(PREVIEW_PATH)))
        except Exception as error:
            print(f'Cannot render: {error}')

for control in controls:
    control.observe(render_preview, names='value')

control_grid = widgets.VBox([
    widgets.HBox([layout, label_mode]),
    widgets.HBox([epsilon]),
    widgets.HBox([beta1, beta2]),
    widgets.HBox([error1, error2, error3]),
    widgets.HBox([z_low, z_high]),
])
display(control_grid, preview_output)
render_preview()

## Export the current view

The button writes PNG, PDF, and plot-ready CSV files using the current controls. PDF output embeds editable TrueType text rather than Type 3 glyphs.

In [ ]:
export_button = widgets.Button(description='Export current view', button_style='primary')
export_status = widgets.Output()

def export_current_view(_):
    with export_status:
        clear_output(wait=True)
        eps_tag = f'{epsilon.value:.4g}'.replace('.', 'p')
        layout_tag = layout.value.replace('-', '_')
        stem = f'iso_error_geometry_{layout_tag}_{label_mode.value}_eps{eps_tag}'
        png_path = REPO_ROOT / 'figures' / 'geometry' / f'{stem}.png'
        pdf_path = REPO_ROOT / 'figures' / 'geometry' / f'{stem}.pdf'
        csv_path = REPO_ROOT / 'tables' / f'{stem}_contours.csv'
        try:
            make_figure(
                **current_parameters(),
                output=png_path,
                pdf_output=pdf_path,
                table_output=csv_path,
            )
            print(f'Saved {png_path}')
            print(f'Saved {pdf_path}')
            print(f'Saved {csv_path}')
        except Exception as error:
            print(f'Cannot export: {error}')

export_button.on_click(export_current_view)
display(export_button, export_status)